# Heatwaves and excess mortality: a piecewise ITS analysis (UK)

:::{note}
**Draft scaffold.** Work-in-progress placeholder. See the "Plan" section for the intended narrative, data, and method. Nothing here is final.
:::

## Motivation

The UK is unusually exposed to heat-related mortality: an old, poorly-insulated housing stock, near-zero domestic air conditioning, and a population unaccustomed to extreme heat. Homes have thermal mass, so during a sustained hot spell the indoor environment keeps warming even after the outdoor peak, and deaths climb over the course of the episode rather than tracking a single hot day.

The timely hook is the **July 2022 heatwave**, when the UK passed 40°C for the first time on record and the ONS attributed hundreds of excess deaths to the heat episodes that summer.

This is a natural companion to the existing [Excess deaths due to COVID-19](its_covid.ipynb) notebook: same interrupted-time-series machinery, but the interruption is a heatwave rather than a pandemic.

## Plan (to build out)

**Estimand.** Excess mortality during and after a heatwave, relative to the counterfactual mortality that the pre-heatwave trend and seasonal baseline imply. This is a clean quasi-experiment: the heatwave is an intervention at a **known date**, and next week's deaths do not cause this week's weather, so there is no reverse causation.

**Why piecewise ITS.** A heatwave is not an instantaneous shock. [Piecewise interrupted time series (segmented regression)](piecewise_its_pymc.ipynb) is built for exactly this: it estimates a **level change** (the acute jump in deaths as the heat hits) *and* a **slope change** (the build-up as homes heat up over successive days, then the compensating dip afterwards from short-term mortality displacement / "harvesting"). The level-plus-slope decomposition is how the "houses warm up" thesis shows up in the model, without needing a continuous dose-response curve.

**Method sketch.**
1. Weekly expected-mortality baseline: cyclic seasonal term + slow long-term/ageing trend (population offset per age band), in the spirit of the COVID notebook.
2. Frame the major heatwaves (2018, 2019, 2022) as repeated known-date interruptions and fit a piecewise ITS with `causalpy`, reading off level and slope effects per episode.
3. Bring the engineered daily heat-load in as a predictor within the model: an exponentially-weighted (thermal-mass) temperature, consecutive-hot-day count, and an apparent-temperature/heat-index term that folds in humidity.
4. Quantify excess deaths per episode as the counterfactual gap, with posterior uncertainty.
5. Falsification: placebo-in-time interruptions on non-heatwave dates should show no effect.

**Headline figure.** Observed weekly deaths, the counterfactual band, heatwave windows shaded, and the excess quantified per episode.

**Scope note.** The graded, whole-year temperature–mortality dose-response (the V-shaped curve with cold and heat arms) is a different, non-quasi-experimental question. It is deliberately out of scope here; this notebook is about discrete heatwave episodes as interruptions, which is what CausalPy is for.

## Data (sourced, to be assembled)

- **Deaths (weekly, ONS).** `weekly-deaths-age-sex` via the ONS API: England & Wales, 5-year age bands to 100+, **occurrence** dates (avoids bank-holiday registration artifacts), 2010–present. Full history is split across editions (`2010-19`, `covid-19`, yearly), so it needs stitching into one series. Age-band population estimates provide the denominator/offset.
- **Weather (daily, Open-Meteo ERA5, CC-BY).** Mean/max/min 2 m temperature, relative humidity, dewpoint, population-weighted across a handful of English points, then engineered to weekly heat-load features. Confirmed to resolve the 19 Jul 2022 spike (~37.6°C at a central point).

Both fetch cleanly; the packaged monthly `"covid"` dataset below is only a placeholder so this scaffold runs.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import causalpy as cp

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'
seed = 42

In [ ]:
# Placeholder: packaged monthly E&W deaths + temperature, so the scaffold runs.
# To be replaced with the weekly ONS occurrence series + engineered daily weather.
df = cp.load_data("covid")
df["date"] = pd.to_datetime(df["date"])

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["date"], df["deaths"])
ax[0].set(ylabel="deaths / month")
ax[1].plot(df["date"], df["temp"], color="C1")
ax[1].set(ylabel="mean temp (°C)", xlabel="date");

## Next steps

- [ ] Stitch the ONS weekly age-banded occurrence series (2010–present) and add age-band population offsets.
- [ ] Pull population-weighted daily English weather; engineer weekly heat-load (EWMA temp, consecutive hot days, heat index with humidity).
- [ ] Fit a piecewise ITS with heatwaves (2018/2019/2022) as known-date interruptions; read off level and slope effects.
- [ ] Quantify excess deaths per episode with uncertainty; build the headline figure.
- [ ] Placebo-in-time falsification on non-heatwave dates.
- [ ] Register a thumbnail and finalise the gallery entry.